# NB78 — Option C smoke test: drive CLRKDNet `CLRHead` from our pipeline

**Goal**: confirm that CLRKDNet's actual `CLRHead` (imported from `external_repos/CLRKDNet-master/`) can be instantiated and forwarded using our backbone's 3-level feature pyramid. If this passes, Option A (execute the full Path-3 plan in RMT-PPAD's repo) or Option B (adapt the plan into our existing codebase) becomes viable.

**Scope** (intentionally narrow):
- Build a joint model with `lane_head.type='clrnet_official'`.
- Forward a random `(2, 3, 384, 640)` tensor.
- Assert lane-head output shapes: `cls_logits` `(2, 192)`, `coord_pred` `(2, 192, 72, 2)`, `lane_param` `(2, 192, 4)`.
- No training, no real data, no loss path. Eval-mode forward only.

**Dependency risk**: CLRHead imports `mmcv.cnn.ConvModule` and `clrkd.ops.nms`. We monkey-patch the NMS C++ extension to a Python stub (we never invoke it in the smoke test). `mmcv` must be pip-installed; cell 2 handles that.

**Decision tree based on outcome**:
- Exit 0 → smoke passed → tell me which path to commit to (Option A vs B) and I'll write the next concrete plan.
- Exit 1 → import failed → most likely missing `mmcv`. Cell 2 should have installed it; if it didn't, the install logs tell us why.
- Exit 2 → shape assertion failed → my wrapper has a translation bug between CLRHead's 78-D format and our pipeline's dict.
- Exit 3 → forward crashed → likely a tensor-shape mismatch between our backbone's output channels and the adapters I wired up.


### Run mode
1. Cell 2 mounts Drive, sets `REPO_ROOT`, and `pip install`s mmcv.
2. Cell 3 runs the smoke test script and prints its full output.
3. Cell 4 summarizes the exit code in plain English.

Expected wall-clock: ~3-5 minutes (mostly the mmcv install).

In [3]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Install mmcv. CLRHead uses mmcv.cnn.ConvModule. We try mmcv-lite
# first (smaller, no CUDA ops needed for ConvModule) and fall back to
# full mmcv if needed.
try:
    import mmcv  # noqa: F401
    print('[ok] mmcv already installed')
except ImportError:
    print('Installing mmcv...')
    rc = subprocess.call([sys.executable, '-m', 'pip', 'install', '-q', 'mmcv-lite'])
    if rc != 0:
        print('mmcv-lite failed; trying full mmcv')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mmcv'])
    import mmcv  # noqa: F401
    print('[ok] mmcv installed:', mmcv.__version__)

# Also install scipy + opencv-python-headless if not present (already
# on Colab usually, but doesn't hurt to confirm).
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'pyyaml', 'scipy', 'opencv-python-headless'])

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)
print('repo:', REPO_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[ok] mmcv already installed
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [4]:
from pathlib import Path
import os, sys

SCRIPT = 'stage2/scripts/smoke_test_clrnet_head.py'
LOG_FILE = os.path.join(LOG_DIR, 'optionC_smoke_clrnet_head.log')

# Use check=False so we capture the exit code rather than raising.
import subprocess
print('Running:', SCRIPT)
print('Log:', LOG_FILE)
with open(LOG_FILE, 'w', encoding='utf-8') as logf:
    proc = subprocess.run(
        [sys.executable, '-u', SCRIPT],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    logf.write(proc.stdout)
print(proc.stdout)
print(f'\n[exit_code] {proc.returncode}')
SMOKE_EXIT = proc.returncode

Running: stage2/scripts/smoke_test_clrnet_head.py
Log: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/optionC_smoke_clrnet_head.log
=== Option C smoke test: CLRKDNet CLRHead via our pipeline ===
project_root: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane  (sys.path[0])
repo_root:    /content/drive/MyDrive/EcoCAR     (contains external_repos/)
[OK] VendorCLRNetHead imported. CLRKDNet vendor: /content/drive/MyDrive/EcoCAR/external_repos/CLRKDNet-master
[OK] CLRHead imported: clrkd.models.heads.clr_head.CLRHead
[OK] torch=2.10.0+cu128, cuda_available=True
[OK] joint model built in 0.16s. params=8.24M
[OK] lane_head type: VendorCLRNetHead
     prior_feat_channels=64, num_priors=192, num_points=72, refine_layers=3
[OK] forward pass completed in 1271.8ms on cuda

Lane head output shapes:
  [OK] cls_logits: got (2, 192), expected (2, 192)
  [OK] coord_pred: got (2, 192, 72, 2), expected (2, 192, 72, 2)
  [OK] lane_param: got (2, 192, 4), expected (2, 192, 4)
  coord_pred range: 

In [5]:
if SMOKE_EXIT == 0:
    print('=' * 60)
    print('PASS: CLRKDNet CLRHead can be driven from our backbone.')
    print('Decision needed: tell me which to commit to:')
    print('  Option A -- full Path-3 in external_repos/RMT-PPAD-main/')
    print('  Option B -- adapt Path-3 into yolop_vehicle_lane/stage2/')
    print('=' * 60)
elif SMOKE_EXIT == 1:
    print('=' * 60)
    print('FAIL: import error. Likely fix:')
    print('  - Confirm cell 2 finished `pip install mmcv` without error.')
    print('  - Check the log above for the actual ImportError trace.')
    print('=' * 60)
elif SMOKE_EXIT == 2:
    print('=' * 60)
    print('FAIL: shape assertion. The wrapper translation between')
    print('CLRHead 78-D format and our pipeline dict is wrong.')
    print('Check `vendor_clrnet_head.VendorCLRNetHead.forward()`.')
    print('=' * 60)
elif SMOKE_EXIT == 3:
    print('=' * 60)
    print('FAIL: forward crashed. Likely fix:')
    print('  - Backbone output channels do not match adapter `in_channels`.')
    print('  - CLRHead expects a different number of feature levels.')
    print('  - Check the log above for the full traceback.')
    print('=' * 60)
else:
    print(f'Unexpected exit code: {SMOKE_EXIT}')

PASS: CLRKDNet CLRHead can be driven from our backbone.
Decision needed: tell me which to commit to:
  Option A -- full Path-3 in external_repos/RMT-PPAD-main/
  Option B -- adapt Path-3 into yolop_vehicle_lane/stage2/
